# CSV and JSON: The Data Engineer's Daily Bread

Data arrives from weather stations in 47 countries. Some send CSV. Some send JSON. None of them agree on encoding.

If you have spent any time working with real data, you will already know that CSV and JSON are the two formats you will encounter more than any other. They are everywhere -- exports from databases, API responses, sensor logs, government open data portals, spreadsheets saved by well-meaning colleagues. They are simple, human-readable, and universally supported.

They are also, quietly, a minefield.

CSV has no formal standard that everyone follows. Delimiters change between countries. Quoting rules vary. And encoding -- the way characters are stored as bytes -- is a source of bugs that can silently corrupt your data without you ever noticing.

JSON is more structured, but real-world JSON is often deeply nested, making it awkward to pull into the flat tables that pandas (and SQL) prefer.

In this notebook, we will work through both formats using real-ish climate data. By the end, you will be able to handle the edge cases that trip up most beginners -- and a fair few experienced engineers.

## Setup

In [ ]:
import pandas as pd
import json

## Part 1: Reading CSV -- The Easy Case

Let's start with something that just works. We have a CSV file of weather station data from around the world -- station IDs, names, locations, and average measurements.

In [ ]:
df = pd.read_csv("../data/weather_stations.csv")
df.head(10)

In [ ]:
df.shape

In [ ]:
df.dtypes

500 rows, sensible column names, correct dtypes. pandas has done its job. The file uses commas as delimiters, UTF-8 encoding, and has no surprises.

This is the happy path. Enjoy it while it lasts.

## Part 2: CSV Edge Cases

The trouble with CSV is that "CSV" is more of a loose suggestion than a specification. Different tools produce different dialects. Here are the common variations you will encounter.

### Semicolons instead of commas

Many European countries use the comma as a decimal separator (so 3,14 instead of 3.14). When those countries export CSV files, they use semicolons as the field delimiter instead, to avoid ambiguity. Germany, France, Italy, Spain, Brazil -- all common offenders.

Let's simulate this. We will write out a small sample using semicolons, then try to read it back.

In [ ]:
# Write a small sample with semicolons
sample = df.head(5)
sample.to_csv("/tmp/stations_semicolon.csv", sep=";", index=False)

# Now try reading it with default settings
broken = pd.read_csv("/tmp/stations_semicolon.csv")
broken.head()

That is a mess. pandas assumed commas, so the entire row ended up in one column. The fix is straightforward:

In [ ]:
fixed = pd.read_csv("/tmp/stations_semicolon.csv", sep=";")
fixed.head()

### Tab-separated values (TSV)

Another common variant. Often seen in bioinformatics, legacy systems, and data exported from databases. Same fix -- just change the separator.

In [ ]:
sample.to_csv("/tmp/stations_tabs.tsv", sep="\t", index=False)

tsv = pd.read_csv("/tmp/stations_tabs.tsv", sep="\t")
tsv.head()

### Quoting and embedded commas

What happens when a field itself contains a comma? Proper CSV wraps that field in double quotes. But not all producers do this correctly.

In [ ]:
# A CSV string with quoted fields containing commas
tricky_csv = """name,description,value
"Station Alpha","Located in Zurich, Switzerland",42.5
"Station Beta","Near the coast, facing west",38.1
"Station Gamma","High altitude, remote",15.7
"""

from io import StringIO
pd.read_csv(StringIO(tricky_csv))

pandas handles this correctly because it respects the quoting rules by default. If you ever encounter a file where quoting is broken, you can control the behaviour with the `quoting` and `quotechar` parameters.

The key takeaways for CSV edge cases:

| Problem | Solution |
|---|---|
| Semicolons instead of commas | `sep=";"` |
| Tabs instead of commas | `sep="\t"` |
| Fields containing the delimiter | Should be quoted -- pandas handles this |
| Different quote character | `quotechar="'"` |
| No header row | `header=None` |
| Skip initial rows | `skiprows=n` |

## Part 3: The Encoding Bomb

This is the one that catches people. It can silently corrupt your data, and you might not notice until someone complains that their city name looks like hieroglyphics.

### What is encoding?

Computers store text as numbers. An encoding is the lookup table that maps characters to numbers. The two encodings you will encounter most often:

- **UTF-8**: The modern standard. Can represent every character in every language. Variable-length (ASCII characters use 1 byte, accented characters use 2, emoji use 4). This is what you should use for everything.
- **Latin-1** (ISO-8859-1): An older encoding that covers Western European languages. Fixed at 1 byte per character. Cannot represent Chinese, Arabic, emoji, or many other scripts.

The problem: a file does not announce its encoding. There is no reliable way to auto-detect it. If you read a Latin-1 file as UTF-8, or vice versa, the accented characters turn into garbage. This is called **mojibake**.

### Let's see it happen

We have the same weather station data saved in two files:
- `weather_stations.csv` -- encoded in UTF-8
- `weather_stations_latin1.csv` -- encoded in Latin-1

The station names include cities like Zurich, Sao Paulo, Malmo, Dusseldorf -- all with accented characters.

Let's read the Latin-1 file with the default encoding (UTF-8):

In [ ]:
# Read the Latin-1 file with the WRONG encoding (UTF-8, the default)
try:
    wrong = pd.read_csv("../data/weather_stations_latin1.csv")
    print(wrong["station_name"].unique()[:10])
except UnicodeDecodeError as e:
    print(f"UnicodeDecodeError: {e}")

Depending on the exact bytes, you will either get a `UnicodeDecodeError` (which is actually the *good* outcome, because at least you know something went wrong), or you will get garbled text -- mojibake -- where the accented characters have been misinterpreted.

Let's force it to read even if there are errors, so we can see the damage:

In [ ]:
wrong = pd.read_csv("../data/weather_stations_latin1.csv", encoding="utf-8", errors="replace")
wrong[wrong["station_name"].str.contains("\ufffd", na=False)][["station_id", "station_name"]].drop_duplicates().head(10)

See those replacement characters? That is what happens when bytes from one encoding are interpreted through another. The city names are ruined.

Now let's read it correctly:

In [ ]:
correct = pd.read_csv("../data/weather_stations_latin1.csv", encoding="latin-1")
correct["station_name"].unique()[:15]

Zurich, Sao Paulo, Malmo, Dusseldorf -- all correct. The only difference was specifying `encoding="latin-1"`.

### Practical advice on encoding

1. **Always save your own files as UTF-8.** There is no reason to use anything else in 2024.
2. **When reading files from others**, try UTF-8 first. If that fails or produces garbled text, try `latin-1`.
3. **If you are getting data from a European government agency or an older system**, there is a good chance it is Latin-1.
4. **The `chardet` library** can guess the encoding, but it is not always right. Treat it as a hint, not a guarantee.

## Part 4: JSON Basics

JSON (JavaScript Object Notation) is the other format you will see everywhere. APIs return it. Configuration files use it. NoSQL databases store it.

Python has a built-in `json` module. Let's start with the basics.

In [ ]:
# Parse a JSON string
raw = '{"station_id": "WS001", "name": "Zurich Central", "temp_c": 15.2}'
data = json.loads(raw)
print(type(data))
print(data["name"])

In [ ]:
# Convert Python dict back to JSON string
back_to_json = json.dumps(data, indent=2)
print(back_to_json)

The key functions:

| Function | Input | Output | Use case |
|---|---|---|---|
| `json.loads(s)` | JSON string | Python dict/list | Parse API responses |
| `json.dumps(obj)` | Python dict/list | JSON string | Prepare data for sending |
| `json.load(f)` | File object | Python dict/list | Read JSON files |
| `json.dump(obj, f)` | Python dict/list | Writes to file | Save JSON files |

The difference between `loads`/`dumps` and `load`/`dump` is simply whether you are working with a string or a file. The `s` stands for "string".

### Reading a JSON file

In [ ]:
with open("../data/ocean_buoys.json", "r") as f:
    buoys = json.load(f)

print(f"Type: {type(buoys)}")
print(f"Number of readings: {len(buoys)}")
print(f"\nFirst reading:")
print(json.dumps(buoys[0], indent=2))

## Part 5: The Nesting Problem

Look at that JSON structure. Each buoy reading has nested objects: `location` contains `lat`, `lon`, and `name`. `measurements` contains `water_temp_c`, `wave_height_m`, and a further nested `wind` object with `speed_ms` and `direction_deg`.

This is very common in real-world JSON. APIs love nesting. But pandas DataFrames are flat -- rows and columns. We need to flatten this nested structure.

### The manual way (painful)

You *could* write a list comprehension to pull out every field by hand:

In [ ]:
# The painful manual approach
manual_rows = []
for b in buoys:
    manual_rows.append({
        "buoy_id": b["buoy_id"],
        "lat": b["location"]["lat"],
        "lon": b["location"]["lon"],
        "location_name": b["location"]["name"],
        "water_temp_c": b["measurements"]["water_temp_c"],
        "wave_height_m": b["measurements"]["wave_height_m"],
        "wave_period_s": b["measurements"]["wave_period_s"],
        "wind_speed_ms": b["measurements"]["wind"]["speed_ms"],
        "wind_direction_deg": b["measurements"]["wind"]["direction_deg"],
        "timestamp": b["timestamp"],
        "status": b["status"],
    })

df_manual = pd.DataFrame(manual_rows)
df_manual.head()

That works, but it is tedious. You have to know the exact structure, type every key by hand, and if the structure changes you have to rewrite it all. For deeply nested JSON with many fields, this approach does not scale.

### The better way: `pd.json_normalize()`

pandas has a function designed specifically for this problem. It recursively flattens nested dictionaries into columns, using dot notation for nested keys.

In [ ]:
df_buoys = pd.json_normalize(buoys)
df_buoys.head()

In [ ]:
df_buoys.columns.tolist()

Look at those column names. `location.lat`, `location.lon`, `measurements.water_temp_c`, `measurements.wind.speed_ms` -- it has walked through the entire nested structure and created a flat column for each leaf value, using dots to show the path.

One line of code replaced that entire manual extraction loop.

### Controlling the separator

If you prefer underscores to dots (which is more SQL-friendly), you can change the separator:

In [ ]:
df_buoys_underscore = pd.json_normalize(buoys, sep="_")
df_buoys_underscore.columns.tolist()

### Quick analysis of the buoy data

Now that we have flat data, we can use all the usual pandas operations.

In [ ]:
df_buoys_underscore.describe()

In [ ]:
# Average water temperature by buoy location
df_buoys_underscore.groupby("location_name")["measurements_water_temp_c"].mean().sort_values(ascending=False).head(10)

In [ ]:
# Status counts
df_buoys_underscore["status"].value_counts()

## Part 6: Reading JSON directly with pandas

For simple cases, you can also use `pd.read_json()` directly:

In [ ]:
# This works but does NOT flatten nested structures
df_simple = pd.read_json("../data/ocean_buoys.json")
df_simple.head(3)

Notice that the `location` and `measurements` columns contain dictionaries. `pd.read_json()` does not flatten nested objects -- it just puts them in as dict values. For nested JSON, you almost always want `json.load()` followed by `pd.json_normalize()` instead.

## Summary

| Task | Tool | Key parameters |
|---|---|---|
| Read a CSV | `pd.read_csv()` | `sep`, `encoding`, `header`, `skiprows` |
| Handle non-UTF-8 encoding | `pd.read_csv(encoding="latin-1")` | Also try `cp1252` for Windows files |
| Parse JSON string | `json.loads()` | |
| Read JSON file | `json.load(f)` | |
| Flatten nested JSON | `pd.json_normalize()` | `sep` for column name separator |
| Quick JSON read (flat only) | `pd.read_json()` | Does not flatten nested objects |

---

## Exercises

### Exercise 1: Parse a tricky CSV

The following CSV string uses semicolons as delimiters, has a header row, and contains quoted fields with embedded semicolons. Parse it into a DataFrame.

```
id;city;description;temp
1;"Berlin";"Cloudy; cold front approaching";3.2
2;"Rome";"Sunny; warm for January";14.8
3;"Oslo";"Snow; heavy in the north";-5.1
```

In [ ]:
# Your code here
tricky = """id;city;description;temp
1;"Berlin";"Cloudy; cold front approaching";3.2
2;"Rome";"Sunny; warm for January";14.8
3;"Oslo";"Snow; heavy in the north";-5.1
"""


### Exercise 2: Encoding detective

Read `weather_stations_latin1.csv` with the correct encoding. Then filter to find all stations in Sweden and display their names, countries, and average temperatures. Verify the station names display correctly (no garbled characters).

In [ ]:
# Your code here


### Exercise 3: Flatten the buoy data

Using `pd.json_normalize()` with the underscore separator:

1. Load the ocean buoys JSON file
2. Flatten it into a DataFrame
3. Find the buoy location with the highest average wave height
4. Find all readings where wind speed exceeded 15 m/s

In [ ]:
# Your code here


### Exercise 4: Build a JSON record

Create a Python dictionary representing a weather station reading with this structure:

```json
{
  "station_id": "WS099",
  "name": "Edinburgh Castle",
  "location": {"lat": 55.9486, "lon": -3.1999},
  "readings": {
    "temperature_c": 8.5,
    "humidity_pct": 78.2,
    "wind": {"speed_ms": 12.1, "direction": "NW"}
  },
  "timestamp": "2024-03-15T10:00:00Z"
}
```

Convert it to a JSON string with `json.dumps()`, then flatten it into a single-row DataFrame using `pd.json_normalize()`.

In [ ]:
# Your code here
